In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
os.environ["BIRDDOG_NOCODB_ENV"] = "LOCAL"
print(f"using {os.environ.get('BIRDDOG_NOCODB_ENV', 'LOCAL')} nocodb")


using LOCAL nocodb


In [3]:
from birddog.database import Database

2026-08-29 07:34:18,880 [INFO] Using LOCAL nocodb api: http://localhost:8080


In [4]:
db = Database()

2026-08-29 07:34:19,431 [INFO] creating NocoDBDatabase(host=http://localhost:8080, base_id=p79fvr9cjqgpv5n) instance
2026-08-29 07:34:19,456 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight  tgt_rps   budget_s
  ------------------------------------------------------------------------------------------------------
  localhost:api                          5.00    40.09     7.00       0.00            4        -          -


In [21]:
docs, cursor = db.scan(
    "FTP Repo", 
    fields=["path", "size"], 
    view_name="PDF",
    limit=100)

2026-08-29 07:58:07,113 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight  tgt_rps   budget_s
  ------------------------------------------------------------------------------------------------------
  localhost:api                         11.50     0.05     7.00       0.00            4        -          -


In [6]:
docs[0]

{'Id': 47120, 'path': '/Makarov/Lviv/701-1-122.pdf', 'size': 565201587}

In [7]:
def find_size_match(db, size):
    return db.scan_all(
        "Documents",
        where=("byte_size", "eq", size),
        fields="url")

In [8]:
find_size_match(db, docs[0]["size"])

2026-08-29 07:48:17,532 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight  tgt_rps   budget_s
  ------------------------------------------------------------------------------------------------------
  localhost:api                          6.00     0.00     7.00       0.00            4        -          -


[]

In [10]:
for rec in docs[:50]:
    m = find_size_match(db, rec["size"])
    if m:
        print("match", rec, m)

2026-08-29 07:49:18,172 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight  tgt_rps   budget_s
  ------------------------------------------------------------------------------------------------------
  localhost:api                          6.50     0.49     7.00       0.00            4        -          -
match {'Id': 45199, 'path': '/Kyiv/5-4-23/117-1-52.pdf', 'size': 19009335} [{'Id': 8979, 'url': 'https://commons.wikimedia.org/wiki/File:ДАЧкО_117-1-52_Листування_з_директором_училищ_Київської_губернії_щодо_стягнення_свічкового_збору_з..._(1845-1846).pdf'}]


In [11]:
len(docs)

100

In [41]:
def check_size_matches(db, docs):
    sizes = list({str(r["size"]) for r in docs})
    matches = db.scan_all(
        "Documents",
        where=("byte_size", "in", sizes),
        fields=["url", "byte_size"],
    )
    matches_by_size = { }
    for rec in matches:
        size = rec["byte_size"]
        entry = matches_by_size.get(size, [])
        entry.append({
            "Id": rec["Id"],
            "url": rec["url"],
        })
        matches_by_size[size] = entry
    result = []
    for doc in docs:
        match = matches_by_size.get(doc["size"])
        if match:
            result.append({
                "Id": doc["Id"], 
                "path": doc["path"],
                "match": match
            })
    return result

In [42]:
check_size_matches(db, docs)

[{'Id': 44371,
  'path': '/Zhitomir/ProskurovKhmelnitski/DAHMO-R-6447-1-152.pdf',
  'match': [{'Id': 846,
    'url': 'https://commons.wikimedia.org/wiki/File:ДАХмО_Р-6447-1-152_Книга_реєстрації_актів_про_народження_Том_V_(1945).pdf'},
   {'Id': 862,
    'url': 'https://uk.wikisource.org/wiki/File:ДАХмО_Р-6447-1-152_Книга_реєстрації_актів_про_народження_Том_V_(1945).pdf'}]},
 {'Id': 44372,
  'path': '/Zhitomir/ProskurovKhmelnitski/DAHMO-R-6447-1-155.pdf',
  'match': [{'Id': 849,
    'url': 'https://commons.wikimedia.org/wiki/File:ДАХмО_Р-6447-1-155_Книга_реєстрації_актів_про_смерть_Том_III_(1945).pdf'},
   {'Id': 852,
    'url': 'https://uk.wikisource.org/wiki/File:ДАХмО_Р-6447-1-155_Книга_реєстрації_актів_про_смерть_Том_III_(1945).pdf'}]},
 {'Id': 44373,
  'path': '/Zhitomir/ProskurovKhmelnitski/DAHMO-R-6447-1-158.pdf',
  'match': [{'Id': 854,
    'url': 'https://uk.wikisource.org/wiki/File:ДАХмО_Р-6447-1-158_Книга_реєстрації_актів_про_шлюб_Том_I_(1945).pdf'},
   {'Id': 866,
    'url':

In [43]:
cursor = None
matches = []
while True:
    print(cursor)
    docs, cursor = db.scan(
        "FTP Repo", 
        fields=["path", "size"], 
        view_name="PDF",
        limit=500,
        cursor=cursor)
    matches.extend(check_size_matches(db, docs))
    if not cursor:
        break

None
2026-08-29 08:18:57,695 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight  tgt_rps   budget_s
  ------------------------------------------------------------------------------------------------------
  localhost:api                         15.50     0.01     7.00       0.00            4        -          -
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
8500
9000
9500
10000
10500
11000
11500
12000
12500
13000
13500
14000
14500
15000
15500
16000
16500
17000
17500
18000
18500
19000
19500
20000
20500
21000
21500
22000
22500
23000
23500
24000
24500
25000
25500
26000
26500
27000
27500
28000
28500
29000
29500
30000
30500
31000
31500
32000
32500
33000
33500
34000
34500
35000
35500
36000
36500
37000
37500
38000
38500
39000
39500
40000
40500
41000
41500
42000
42500
43000
43500
44000


In [44]:
len(matches)

4646

In [45]:
matches[0]

{'Id': 45199,
 'path': '/Kyiv/5-4-23/117-1-52.pdf',
 'match': [{'Id': 8979,
   'url': 'https://commons.wikimedia.org/wiki/File:ДАЧкО_117-1-52_Листування_з_директором_училищ_Київської_губернії_щодо_стягнення_свічкового_збору_з..._(1845-1846).pdf'}]}